In [1]:
SOURCE_INGESTED_ON = "2026-10-13"

MART_PATH = (
    "s3://mart/seoul_bike_sharing_demand/"
    f"source_ingested_on={SOURCE_INGESTED_ON}/"
    "hourly_rentals.parquet"
)

print("MART:", MART_PATH)

MART: s3://mart/seoul_bike_sharing_demand/source_ingested_on=2026-10-13/hourly_rentals.parquet


In [2]:
import os

import duckdb
from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv(".env", usecwd=True)

if not env_path:
    raise FileNotFoundError("Файл .env не найден")

load_dotenv(env_path)

connection = duckdb.connect()

connection.execute("INSTALL httpfs")
connection.execute("LOAD httpfs")

s3_endpoint = os.environ["S3_ENDPOINT"]
s3_access_key = os.environ["S3_ACCESS_KEY"]
s3_secret_key = os.environ["S3_SECRET_KEY"]

connection.execute(f"SET s3_endpoint='{s3_endpoint}'")
connection.execute("SET s3_url_style='path'")
connection.execute("SET s3_use_ssl=false")
connection.execute("SET s3_region='us-east-1'")
connection.execute(f"SET s3_access_key_id='{s3_access_key}'")
connection.execute(f"SET s3_secret_access_key='{s3_secret_key}'")

print("DuckDB подключён к SeaweedFS")

DuckDB подключён к SeaweedFS


In [3]:
connection.execute(
    f"""
    CREATE OR REPLACE TEMP VIEW mart_source AS
    SELECT *
    FROM read_parquet('{MART_PATH}')
    """
)

print("mart_source создано")

mart_source создано


In [4]:
connection.sql(
    """
    SELECT COUNT(*) AS rows_count
    FROM mart_source
    """
).show()

┌────────────┐
│ rows_count │
│   int64    │
├────────────┤
│       8760 │
└────────────┘



In [5]:
connection.sql(
    """
    SELECT *
    FROM mart_source
    ORDER BY date, hour
    LIMIT 5
    """
).show()

┌────────────┬───────┬───────┬─────────────┬───────┬─────────┬────────────┬─────────────────┬───────────────────┬───────────────────────────────┬───────────────────────────────────────────┬────────────────────┐
│    date    │ year  │ month │ day_of_week │ hour  │ season  │  holiday   │ functioning_day │ rented_bike_count │ quality_flag_invalid_humidity │ quality_flag_functioning_day_inconsistent │ source_ingested_on │
│    date    │ int32 │ int32 │    int32    │ int64 │ varchar │  varchar   │     boolean     │       int64       │            boolean            │                  boolean                  │        date        │
├────────────┼───────┼───────┼─────────────┼───────┼─────────┼────────────┼─────────────────┼───────────────────┼───────────────────────────────┼───────────────────────────────────────────┼────────────────────┤
│ 2017-12-01 │  2017 │    12 │           5 │     0 │ Winter  │ No Holiday │ true            │               254 │ false                         │ false     

In [6]:
connection.sql(
    """
    SELECT
        COUNT(*) AS observation_hours,

        COUNT(*) FILTER (
            WHERE functioning_day IS TRUE
        ) AS functioning_hours,

        SUM(rented_bike_count) FILTER (
            WHERE functioning_day IS TRUE
        ) AS total_rentals,

        ROUND(
            AVG(rented_bike_count) FILTER (
                WHERE functioning_day IS TRUE
            ),
            2
        ) AS avg_rented_bikes,

        MEDIAN(rented_bike_count) FILTER (
            WHERE functioning_day IS TRUE
        ) AS median_rented_bikes

    FROM mart_source
    """
).show()

┌───────────────────┬───────────────────┬───────────────┬──────────────────┬─────────────────────┐
│ observation_hours │ functioning_hours │ total_rentals │ avg_rented_bikes │ median_rented_bikes │
│       int64       │       int64       │    int128     │      double      │       double        │
├───────────────────┼───────────────────┼───────────────┼──────────────────┼─────────────────────┤
│              8760 │              8465 │       6172314 │           729.16 │               542.0 │
└───────────────────┴───────────────────┴───────────────┴──────────────────┴─────────────────────┘



## Основная группировка

Основная группировка `metrics_main` предназначена для сравнения почасового спроса на велосипеды по сезону, праздничному статусу и часу суток.

Одна строка результата соответствует одному сочетанию:

`season + holiday + hour`.

Для каждой группы рассчитываются:
- общее количество часовых наблюдений;
- количество часов, когда сервис функционировал;
- суммарное количество аренд в функционирующие часы;
- среднее количество аренд в функционирующие часы;
- медианное количество аренд в функционирующие часы;
- доля группы в общем количестве аренд.

Среднее и медиана не включают часы с `functioning_day = false`.

In [7]:
connection.execute(
    """
    CREATE OR REPLACE TEMP TABLE metrics_main AS

    WITH grouped AS (
        SELECT
            season,
            holiday,
            hour,

            COUNT(*) AS observation_hours,

            COUNT(*) FILTER (
                WHERE functioning_day IS TRUE
            ) AS functioning_hours,

            SUM(rented_bike_count) FILTER (
                WHERE functioning_day IS TRUE
            ) AS total_rentals,

            ROUND(
                AVG(rented_bike_count) FILTER (
                    WHERE functioning_day IS TRUE
                ),
                2
            ) AS avg_rented_bikes,

            MEDIAN(rented_bike_count) FILTER (
                WHERE functioning_day IS TRUE
            ) AS median_rented_bikes

        FROM mart_source

        GROUP BY
            season,
            holiday,
            hour
    )

    SELECT
        *,

        ROUND(
            100.0 * total_rentals
            / NULLIF(
                SUM(total_rentals) OVER (),
                0
            ),
            4
        ) AS rental_share_pct

    FROM grouped
    """
)

print("metrics_main создано")

metrics_main создано


In [8]:
connection.sql(
    """
    SELECT COUNT(*) AS metrics_main_rows
    FROM metrics_main
    """
).show()

┌───────────────────┐
│ metrics_main_rows │
│       int64       │
├───────────────────┤
│               192 │
└───────────────────┘



In [9]:
connection.sql(
    """
    SELECT *
    FROM metrics_main
    ORDER BY
        season,
        holiday,
        hour
    """
).show()

┌─────────┬────────────┬───────┬───────────────────┬───────────────────┬───────────────┬──────────────────┬─────────────────────┬──────────────────┐
│ season  │  holiday   │ hour  │ observation_hours │ functioning_hours │ total_rentals │ avg_rented_bikes │ median_rented_bikes │ rental_share_pct │
│ varchar │  varchar   │ int64 │       int64       │       int64       │    int128     │      double      │       double        │      double      │
├─────────┼────────────┼───────┼───────────────────┼───────────────────┼───────────────┼──────────────────┼─────────────────────┼──────────────────┤
│ Autumn  │ Holiday    │     0 │                 5 │                 4 │          3098 │            774.5 │               750.0 │           0.0502 │
│ Autumn  │ Holiday    │     1 │                 5 │                 4 │          2360 │            590.0 │               535.0 │           0.0382 │
│ Autumn  │ Holiday    │     2 │                 5 │                 4 │          1768 │            442.0 

In [10]:
connection.sql(
    """
    SELECT
        ROUND(
            SUM(rental_share_pct),
            2
        ) AS total_share_pct
    FROM metrics_main
    """
).show()

┌─────────────────┐
│ total_share_pct │
│     double      │
├─────────────────┤
│           100.0 │
└─────────────────┘



## Уточняющая группировка

Таблица `metrics_detail` уточняет основную группировку с помощью признака `day_of_week`.

Одна строка результата соответствует сочетанию:

`season + holiday + day_of_week + hour`.

Для каждой группы рассчитываются те же показатели, что и в основной группировке:
- количество часовых наблюдений;
- количество функционирующих часов;
- суммарное количество аренд;
- среднее количество аренд;
- медианное количество аренд.

Среднее, медиана и сумма аренд рассчитываются только для часов, в которые сервис функционировал.

In [11]:
connection.execute(
    """
    CREATE OR REPLACE TEMP TABLE metrics_detail AS

    SELECT
        season,
        holiday,
        day_of_week,
        hour,

        COUNT(*) AS observation_hours,

        COUNT(*) FILTER (
            WHERE functioning_day IS TRUE
        ) AS functioning_hours,

        SUM(rented_bike_count) FILTER (
            WHERE functioning_day IS TRUE
        ) AS total_rentals,

        ROUND(
            AVG(rented_bike_count) FILTER (
                WHERE functioning_day IS TRUE
            ),
            2
        ) AS avg_rented_bikes,

        MEDIAN(rented_bike_count) FILTER (
            WHERE functioning_day IS TRUE
        ) AS median_rented_bikes

    FROM mart_source

    GROUP BY
        season,
        holiday,
        day_of_week,
        hour
    """
)

print("metrics_detail создано")

metrics_detail создано


In [12]:
connection.sql(
    """
    SELECT COUNT(*) AS metrics_detail_rows
    FROM metrics_detail
    """
).show()

┌─────────────────────┐
│ metrics_detail_rows │
│        int64        │
├─────────────────────┤
│                 960 │
└─────────────────────┘



In [13]:
connection.sql(
    """
    SELECT *
    FROM metrics_detail
    ORDER BY
        season,
        holiday,
        day_of_week,
        hour
    """
).show()

┌─────────┬────────────┬─────────────┬───────┬───────────────────┬───────────────────┬───────────────┬──────────────────┬─────────────────────┐
│ season  │  holiday   │ day_of_week │ hour  │ observation_hours │ functioning_hours │ total_rentals │ avg_rented_bikes │ median_rented_bikes │
│ varchar │  varchar   │    int32    │ int64 │       int64       │       int64       │    int128     │      double      │       double        │
├─────────┼────────────┼─────────────┼───────┼───────────────────┼───────────────────┼───────────────┼──────────────────┼─────────────────────┤
│ Autumn  │ Holiday    │           1 │     0 │                 1 │                 1 │           620 │            620.0 │               620.0 │
│ Autumn  │ Holiday    │           1 │     1 │                 1 │                 1 │           426 │            426.0 │               426.0 │
│ Autumn  │ Holiday    │           1 │     2 │                 1 │                 1 │           302 │            302.0 │               

## Отдельные числовые значения

Для будущего анализа распределения подготовлена таблица `rental_distribution_values`.

Она содержит отдельное значение `rented_bike_count` для каждого часа, в который сервис функционировал.

Нефункционирующие часы исключены из этой выборки, поскольку нулевое количество аренд в такие периоды связано с недоступностью сервиса, а не с обычным пользовательским спросом.

In [14]:
connection.execute(
    """
    CREATE OR REPLACE TEMP TABLE rental_distribution_values AS

    SELECT
        rented_bike_count

    FROM mart_source

    WHERE functioning_day IS TRUE
      AND rented_bike_count IS NOT NULL
    """
)

print("rental_distribution_values создано")

rental_distribution_values создано


In [15]:
connection.sql(
    """
    SELECT COUNT(*) AS distribution_values_count
    FROM rental_distribution_values
    """
).show()

┌───────────────────────────┐
│ distribution_values_count │
│           int64           │
├───────────────────────────┤
│                      8465 │
└───────────────────────────┘



In [16]:
connection.sql(
    """
    SELECT *
    FROM rental_distribution_values
    LIMIT 10
    """
).show()

┌───────────────────┐
│ rented_bike_count │
│       int64       │
├───────────────────┤
│               328 │
│               388 │
│               618 │
│               227 │
│               498 │
│               269 │
│               789 │
│               145 │
│               381 │
│               496 │
├───────────────────┤
│      10 rows      │
└───────────────────┘



## Проверки расчётов

### Проверка 1. Сумма размеров основных групп

Так как группы `season + holiday + hour` не пересекаются, сумма `observation_hours` должна совпадать с общим количеством строк витрины.

In [17]:
connection.sql(
    """
    SELECT
        SUM(observation_hours) AS grouped_observation_hours,

        (
            SELECT COUNT(*)
            FROM mart_source
        ) AS total_observation_hours,

        SUM(observation_hours)
        =
        (
            SELECT COUNT(*)
            FROM mart_source
        ) AS check_passed

    FROM metrics_main
    """
).show()

┌───────────────────────────┬─────────────────────────┬──────────────┐
│ grouped_observation_hours │ total_observation_hours │ check_passed │
│          int128           │          int64          │   boolean    │
├───────────────────────────┼─────────────────────────┼──────────────┤
│                      8760 │                    8760 │ true         │
└───────────────────────────┴─────────────────────────┴──────────────┘



### Проверка 2. Число значений, участвующих в метриках

Сумма `functioning_hours` основной группировки должна совпадать с отдельным подсчётом часов, в которые сервис функционировал.

In [18]:
connection.sql(
    """
    SELECT
        SUM(functioning_hours) AS grouped_functioning_hours,

        (
            SELECT COUNT(*)
            FROM mart_source
            WHERE functioning_day IS TRUE
        ) AS actual_functioning_hours,

        SUM(functioning_hours)
        =
        (
            SELECT COUNT(*)
            FROM mart_source
            WHERE functioning_day IS TRUE
        ) AS check_passed

    FROM metrics_main
    """
).show()

┌───────────────────────────┬──────────────────────────┬──────────────┐
│ grouped_functioning_hours │ actual_functioning_hours │ check_passed │
│          int128           │          int64           │   boolean    │
├───────────────────────────┼──────────────────────────┼──────────────┤
│                      8465 │                     8465 │ true         │
└───────────────────────────┴──────────────────────────┴──────────────┘



### Проверка 4. Количество отдельных значений

Число строк в `rental_distribution_values` должно совпадать с количеством функционирующих часов, поскольку для каждого такого часа сохраняется одно значение `rented_bike_count`.

In [19]:
connection.sql(
    """
    SELECT
        (
            SELECT COUNT(*)
            FROM rental_distribution_values
        ) AS distribution_values,

        (
            SELECT COUNT(*)
            FROM mart_source
            WHERE functioning_day IS TRUE
              AND rented_bike_count IS NOT NULL
        ) AS expected_values,

        (
            SELECT COUNT(*)
            FROM rental_distribution_values
        )
        =
        (
            SELECT COUNT(*)
            FROM mart_source
            WHERE functioning_day IS TRUE
              AND rented_bike_count IS NOT NULL
        ) AS check_passed
    """
).show()

┌─────────────────────┬─────────────────┬──────────────┐
│ distribution_values │ expected_values │ check_passed │
│        int64        │      int64      │   boolean    │
├─────────────────────┼─────────────────┼──────────────┤
│                8465 │            8465 │ true         │
└─────────────────────┴─────────────────┴──────────────┘

